# Mobile apps review classification with spaCy and sklearn (Vector Space Model and Logistic Regression)

This notebook illustrates text classification using **Vector Space Model (VSM)** for text representation and **Logistic Regression (LR)** algorithm for classification. <br>

The classification task is classification of mobile app reviews into:
* functional requirements,
* non-functional requirements,
* other (reviews irrelevant from Requirements Engineering perspective)

The dataset used in the notebook originates from the paper [Listening to the Crowd for the Release Planning of Mobile Apps](https://dl.acm.org/doi/abs/10.1109/TSE.2017.2759112). It consists of manually classified 3,000 user reviews of different Android apps.<br>

### Load the required libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn import metrics

from scipy.stats import loguniform, uniform
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from IPython.display import display

import re
import spacy

import json
from pathlib import Path

In [ ]:
RAND_STATE = 1

### Load the data

Loading the data in case of running this code from Google Colab

In [ ]:
# from google.colab import files
#
# data_file = files.upload()
#
# The uploaded object is a dictionary with the filename and the file content as key-value pairs - in this case just one such pair.
# We extract the file name and read in the file content using pandas (to have it in the DataFrame format)
# file_name = list(data_file.keys())[0]
# data = pd.read_csv(file_name)

Create the outcome variable (`cls`) as was done in the previous class:
* functional: BUG, FEATURE
* non-functional: PERFORMANCE, USABILITY, ENERGY, SECURITY
* other: OTHER

As at the previous class, we will include review length and rating as predictors, in addition to the review text. The other variables will be dropped

### Text cleaning (optional)

**Note**: to reduce the complexity of this task, we may skip the text cleaning component as it is not necessary for the text we are currently working with. Still, it is an often used component when working with text collected from the Web, and you are encouraged to explore it.

If (textual) data was collected from web sites, forums, online social networks, and other similar sources, it often needs to be "cleaned" before being sent to spaCy for preprocessing. This cleaning typically includes removal of HTML markup,links, hashtags and the like.

For a refresher on regular expressions, check a cheatsheet such as [this one](https://www.dataquest.io/cheat-sheet/regular-expressions-cheat-sheet/).

To be able to include text cleaning component in the model building pipeline, we will create a custom transformer by subclassing scikit-learn's base class for all transformers ([`TransformerMixin`](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html)). This is mandatory for integrating it into a sklearn Pipeline (only transformers and estimators can be included).<br> This will be the first step in text preprocessing (and the overall pipeline), which will be followed by the 'vectorisation' step.

### Text transformation with the Vector Space Model

#### Text preprocesing (using Spacy)

First, we need to prepare a function for text preprocessing. This will be done in the same way as in the previous class - using spacy.

So, we star by loading spacy's basic English language model, that is, a pipeline of software componets for processing English language texts.

Reminder: if the previous line ends in an error, it is because spacy's `en_core_web_sm` language model is not available. <br>To download it, execute the following line in the command prompt or terminal: <br>
`python -m spacy download en_core_web_sm` <br>

We will have the same function as before for custom preprocessing of app reviews, using spacy's pipeline:

#### Text transformation with TF-IDF vectoriser

Next, we create TF-IDF vectoriser that, for the given set of documents (reviews), builds a vector space model with TF-IDF weights.

This vectoriser is an instance of the scikit-learn's [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) class

We can examine what the transofmed data looks like

### Create and fit text classification pipeline

As before, we will create a review classifier as a sequence ('pipeline') of several components, using the [sklearn's Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

To build a Pipeline, first, we split the data into train/test sets:

As before, we use the [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) class to apply different data transformations to different columns in the dataset.

As the classifier, we will use **Logistic Regression (LR)** classifier (implemented in sklearn's [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) class).

We can now define and fit the classification pipeline:

### Evaluate the model on the test data

First, examine the confusion matrix

Now, we can compute evaluation measures:

Reminder:

*   `support` is the number of occurrences of each class in y_true
*   `macro avg` is the average of unweighted score per label (i.e. class)
*   `weighted avg` is the average of support-weighted score per label (i.e. class):

weighted avg for metric $m$:

$m = \sum_{i=1}^{n} m_i * \frac{support_i}{N}$

You may want to take a look at [this](https://www.evidentlyai.com/classification-metrics/multi-class-metrics) or [this](https://iamirmasoud.com/2022/06/19/understanding-micro-macro-and-weighted-averages-for-scikit-learn-metrics-in-multi-class-classification-with-example/) article for an excellent explanation of how all these different metrics are computed and when they should be used.

Compute also [balanced accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html).

Reminder: balanced accuracy is, in fact, raw accuracy where each sample is weighted according to the inverse prevalence (see the "support" in the table above) of its true class.

### Try to improve the classifer through hyperparameter tuning

We will now try to improve the performance of the classifier by changing the way text transformation is done and how LR algorithm is configured.

Hyperparameters often used for tuning TF-IDF vectorise include:
* `min_df` - minimum document freequency
* `ngram_range` - ngrams that will be used as features
* `max_features` - the nuber of ngrams to keep, sorted based on their TF-IDF values

When it comes to Logistic Regression in sklearn, there are a couple of important hyperparameters (C, penalty, solver). Among these, most often tuned one is `C` (Inverse Regularization Strength), which controls the trade-off between fitting the training data and avoiding overfitting:
* Low C values lead to a stronger regularization. It shrinks coefficients and creates a simpler model (higher bias, lower variance)
* Larger C values imply weaker regularization. It leads to larger coefficients, fitting training data more closely (lower bias, higher variance)

We will try to determine optimal value of the above hyperparameters through *Cross-Validation*, using the [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) class.

**Reminder**: hyperparamter names have to be preceded by the name of the corresponding component (transformer or estimator) in the pipeline; in this case, names of the relevant pipeline components are `vectorizer` and `classifier`. However, as the 'vectorizer' is not a single transformer but may contain multiple transformers (one per column), we need to specify the name of the particular transformer we want to fine-tune - in this case, it is 'word_counts'. So, we have to precede the name of the text transformation hyperparameter with `vectorizer__word_counts__`.

Note: `logspace` is a NumPy function that generates numbers that are evenly spaced on a logarithmic scale, rather than a linear scale. For example, for the log base of 10, `logspace` multiplies each value by 10 to produce the next one; so, each step leads to an order of magnitude increase.

Re-run the pipeline with the optimal hyperparamter values:

Make predictions with the fine-tuned pipeline and evaluate the performance

### Compare the created models

We will now compare the models we have built so far and will also include a dummy classifier to see how much better the models are compared to the most basic classifier (the one that generates predictions by respecting the training set’s class distribution)

The following function calculates macro and weighted average metrics and balanced accuracy based on the given input paramters for one classifier